# 01 · Run the pipeline on a clip from S3

Steps: point the config at your clip → pick ≥ 4 court points on the first frame → check the overlay →
run a short test → run the whole clip → look at the events → upload results to S3.

In [ ]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("working in", REPO)

In [ ]:
from vb.config import load_config
cfg = load_config("configs/default.yaml")
cfg.video.source = "s3://YOUR-BUCKET/volleyball/clip.mp4"      # <- your clip
cfg.output.s3_prefix = "s3://YOUR-BUCKET/volleyball/runs"       # <- where results go (or None)
cfg.output.home_team, cfg.output.away_team = "Home", "Away"

## Pick court points
Show the first frame with a pixel grid, read off the pixel positions of court corners / line intersections you can see,
and pair each with its court position in metres. Named points are in `vb.court.COURT_POINTS`
(near = the end closest to the camera: Y = 0; far end Y = 18; net Y = 9; attack lines Y = 6 and 12).

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from vb.io import fetch
from vb.court import COURT_POINTS
clip = fetch(cfg.video.source, cfg.video.local_cache)
cap = cv2.VideoCapture(clip); cap.set(cv2.CAP_PROP_POS_MSEC, cfg.video.start_s * 1000); ok, first = cap.read(); cap.release()
plt.figure(figsize=(16, 9)); plt.imshow(first[..., ::-1]); plt.grid(True, alpha=.4)
plt.xticks(range(0, first.shape[1], 100)); plt.yticks(range(0, first.shape[0], 50)); plt.title("read pixel coords of visible court points")
print(COURT_POINTS)

In [ ]:
# fill in: (u, v) pixels  ->  name of the court point
picked = {
    # "near_left":  (412, 1010),
    # "near_right": (1530, 1004),
    # "far_left":   (760, 352),
    # "far_right":  (1172, 350),
    # "centre_left": (655, 520),
    # "centre_right": (1262, 516),
}
cfg.court.seed_pixels = [list(map(float, uv)) for uv in picked.values()]
cfg.court.seed_court = [list(COURT_POINTS[k]) for k in picked]
len(picked)

In [ ]:
from vb.court import court_homography, draw_court
H0, inl = court_homography(np.float32(cfg.court.seed_court), np.float32(cfg.court.seed_pixels))
over = draw_court(first.copy(), H0, (0, 255, 255), 3)
plt.figure(figsize=(16, 9)); plt.imshow(over[..., ::-1]); plt.title(f"court overlay, {inl.sum()}/{len(inl)} inliers")

## Short test run (first 20 s), then the whole clip

In [ ]:
from vb.pipeline import run_offline
res = run_offline(cfg, run_name="test20s", max_frames=int(20 * 25))
print(json.dumps(res.summary(), indent=1))
print(json.dumps(res.timings, indent=1))

In [ ]:
res = run_offline(cfg, run_name=Path(cfg.video.source).stem)
print(json.dumps(res.summary(), indent=1))

In [ ]:
import pandas as pd
ev = pd.DataFrame(json.loads((res.out_dir / "events.json").read_text()))
print("rallies:")
for r in res.rallies: print(f"  {r.idx:3d}  {r.start_t:7.1f}s  {r.labels}")
ev.head(30)

In [ ]:
pd.DataFrame(res.slots)

In [ ]:
from vb.dvw import check_with_datavolley
plays, problems = check_with_datavolley(res.dvw_path)
print(len(plays), "rows read by py-datavolley;", len(problems), "problems")
pd.DataFrame(plays)[["code", "team", "player_number", "skill", "evaluation_code", "start_zone"]].head(20)

In [ ]:
# a few rendered frames around the first touches
cap = cv2.VideoCapture(str(res.out_dir / "render.mp4"))
fig, axs = plt.subplots(2, 3, figsize=(18, 7))
for ax, t in zip(axs.ravel(), res.touches[:6]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, t.frame - int(cfg.video.start_s * res.fps_video)); ok, im = cap.read()
    if ok: ax.imshow(im[..., ::-1]); ax.set_title(f"{t.t:.1f}s {t.skill} #{t.number}"); ax.axis("off")

In [ ]:
if cfg.output.s3_prefix:
    from vb.io import upload_dir
    uris = upload_dir(str(res.out_dir), f"{cfg.output.s3_prefix}/{res.out_dir.name}")
    print("\n".join(uris))